# Mi primer recorrido de clasificación: Titanic

Continuamos el EDA: ahora queremos **predecir**, no demostrar qué causó la supervivencia.
Trabajad por parejas. Cada persona ejecuta, explica y anota los checkpoints.
Usad el CSV original `titanic.csv` junto a este cuaderno. No se descarga nada.

El EDA anterior se hizo sobre el conjunto completo: esta evaluación es una **primera demostración**, no una estimación completamente independiente. Para un proyecto real reservaríamos prueba antes de explorar y elegir características.

## 1. Pregunta, datos y variables

¿Podemos predecir `Survived` usando algunas características conocidas? Decid por qué
`Survived` y su posible copia `Survived_label` **no** pueden entrar en X. El CSV no se modifica.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

ruta = Path("titanic.csv")
if not ruta.is_file():
    raise FileNotFoundError("Coloca titanic.csv junto al cuaderno y vuelve a ejecutar.")
df = pd.read_csv(ruta)
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)
print("Tamaño:", df.shape)
display(df.head(3))
assert df.shape == (891, 12)
assert set(df["Survived"].unique()) == {0, 1}


pandas 3.0.6 | scikit-learn 1.9.1
Tamaño: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [2]:
# Pclass se trata como categoría aunque esté representada con números.
numericas = ["Age", "Fare", "SibSp", "Parch"]
categoricas = ["Pclass", "Sex", "Embarked"]
X = df[numericas + categoricas].copy()
y = df["Survived"].copy()
assert {"Survived", "Survived_label"}.isdisjoint(X.columns)
print("Características:", X.columns.tolist())
print("Objetivo:", y.name, "| filas:", len(y))
print("Valores ausentes por columna:")
print(X.isna().sum())


Características: ['Age', 'Fare', 'SibSp', 'Parch', 'Pclass', 'Sex', 'Embarked']
Objetivo: Survived | filas: 891
Valores ausentes por columna:
Age         177
Fare          0
SibSp         0
Parch         0
Pclass        0
Sex           0
Embarked      2
dtype: int64


**Checkpoint 1:** Explica con tus palabras qué son X e y y por qué no incluimos la respuesta en X.
Anota una característica excluida y una razón. La selección es una simplificación, no una regla universal.

**Respuesta (Checkpoint 1):**
- **X** son las características conocidas de cada pasajero (edad, tarifa, familiares a bordo, clase, sexo y puerto de embarque). **y** es lo que queremos predecir: `Survived` (0 = no sobrevivió, 1 = sobrevivió).
- `Survived` no puede estar en X porque es la respuesta: el modelo la «copiaría» y obtendría un acierto perfecto que no serviría para predecir pasajeros nuevos (fuga de información). Lo mismo vale para `Survived_label`, que es una copia con otro formato.
- **Característica excluida:** `Name`, `Ticket` y `Cabin`. `Cabin` tiene muchísimos nulos y `Name`/`Ticket` son identificadores casi únicos que el modelo no puede generalizar sin un tratamiento específico. Es una simplificación para este primer recorrido, no una regla universal.

## 2. Separamos antes de aprender transformaciones

Reservamos el 20 % para prueba. Estratificar mantiene aproximadamente las proporciones
de las clases. La semilla permite repetir esta partición bajo el mismo entorno.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
assert set(X_train.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_test) == len(df)
print("Entrenamiento:", X_train.shape, "| prueba:", X_test.shape)
print("Supervivencia en train:", round(y_train.mean(), 3))


Entrenamiento: (712, 7) | prueba: (179, 7)
Supervivencia en train: 0.383


**Checkpoint 2:** ¿Qué conjunto se usa para aprender? ¿Para qué reservamos el otro?
No calcules medianas, escalas o categorías sobre todo el archivo antes de separar.

**Respuesta (Checkpoint 2):**
- Se aprende con **entrenamiento** (80 %, 712 pasajeros): ahí se ajustan las medianas, las escalas, las categorías y los coeficientes.
- La **prueba** (20 %, 179 pasajeros) se reserva para medir cómo se comporta el modelo con pasajeros que no ha visto. Si calculásemos medianas o escalas con todo el archivo antes de separar, información de prueba se filtraría en el aprendizaje y la evaluación sería demasiado optimista.

## 3. Preparación y modelo juntos

Nulos numéricos: mediana; categóricos: valor más frecuente. Codificamos categorías
y escalamos los números. `Pipeline` ajusta preparación y clasificador solo con train.
La configuración está fijada antes de consultar prueba.

### Dos clasificadores, dos funciones

`DummyClassifier` sirve de **referencia**: predice la clase más frecuente de
entrenamiento sin usar las características. `LogisticRegression` aprende
coeficientes de las características preparadas y estima una probabilidad para
clasificar. Su nombre contiene «regresión», pero aquí resuelve una clasificación
binaria. La exactitud no explica por sí sola qué tipos de errores comete.

Para ampliar, consulta `05b-primeros-clasificadores.md` si el profesor facilita
la guía; en este primer recorrido basta con explicar qué aprende cada uno.


In [4]:
preparacion = ColumnTransformer([
    ("numericas", Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler()),
    ]), numericas),
    ("categoricas", Pipeline([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(handle_unknown="ignore")),
    ]), categoricas),
])
modelo = Pipeline([
    ("preparar", preparacion),
    ("clasificar", LogisticRegression(max_iter=1000, random_state=42)),
])
base = DummyClassifier(strategy="most_frequent")
base.fit(X_train, y_train)
modelo.fit(X_train, y_train)
print("Preparación y modelo ajustados únicamente con entrenamiento.")


Preparación y modelo ajustados únicamente con entrenamiento.


### Observar lo que preparó el pipeline

Esta comprobación incorpora la preparación para ML del apartado 4 de la sesión
Titanic de UD2: **no hay que abrir aquel cuaderno después de este**.
El pipeline ya aprendió las medianas, escalas y categorías al entrenar.
Ahora solo usamos `transform` para inspeccionar cómo deja train y test, sin
volver a aprender nada. Las columnas nuevas proceden de la codificación.


In [5]:
preparador_ajustado = modelo.named_steps["preparar"]
train_preparado = preparador_ajustado.transform(X_train)
test_preparado = preparador_ajustado.transform(X_test)
print("Train preparado:", train_preparado.shape)
print("Test preparado:", test_preparado.shape)
print("Columnas:", preparador_ajustado.get_feature_names_out().tolist())
assert train_preparado.shape[0] == len(X_train)
assert test_preparado.shape[0] == len(X_test)
assert train_preparado.shape[1] == test_preparado.shape[1]
for tabla in (train_preparado, test_preparado):
    valores = tabla.toarray() if hasattr(tabla, "toarray") else np.asarray(tabla)
    assert np.isfinite(valores).all()


Train preparado: (712, 12)
Test preparado: (179, 12)
Columnas: ['numericas__Age', 'numericas__Fare', 'numericas__SibSp', 'numericas__Parch', 'categoricas__Pclass_1', 'categoricas__Pclass_2', 'categoricas__Pclass_3', 'categoricas__Sex_female', 'categoricas__Sex_male', 'categoricas__Embarked_C', 'categoricas__Embarked_Q', 'categoricas__Embarked_S']


**Checkpoint 3:** ¿De qué pasajeros aprende la mediana? ¿Qué hace el pipeline
cuando llega un pasajero de prueba con una categoría no vista?

**Respuesta (Checkpoint 3):**
- La mediana (y la moda para las categorías) se aprenden **solo de los pasajeros de entrenamiento**. En esta partición la mediana de `Age` en train es **28,5** años y la categoría más frecuente de `Embarked` es **S**. Se aplican después también a los nulos de prueba, sin recalcularlas.
- Ante una categoría no vista en entrenamiento, `OneHotEncoder(handle_unknown="ignore")` no falla: deja todas las columnas de esa variable en 0 para ese pasajero.

## 4. Predicción base, modelo y errores

La base predice siempre la clase más frecuente de train. Ambos métodos se evalúan
en los mismos pasajeros reservados. La matriz tiene **filas reales y columnas predichas**;
0 = no sobrevivió, 1 = sobrevivió.

In [6]:
pred_base = base.predict(X_test)
pred = modelo.predict(X_test)
print("Exactitud de la base:", round(accuracy_score(y_test, pred_base), 3))
print("Exactitud del modelo:", round(accuracy_score(y_test, pred), 3))
matriz = confusion_matrix(y_test, pred, labels=[0, 1])
display(pd.DataFrame(matriz, index=["Real 0", "Real 1"],
                     columns=["Predicción 0", "Predicción 1"]))
assert matriz.sum() == len(y_test)


Exactitud de la base: 0.615
Exactitud del modelo: 0.804


,Predicción 0,Predicción 1
Real 0,98,12
Real 1,23,46


**Checkpoint 4:** Localiza un falso positivo y un falso negativo en la matriz.
¿Qué conclusión permite la comparación con la base? ¿Por qué la exactitud sola
no basta para garantizar que el modelo sea útil o justo?

No ajustéis el modelo mirando repetidamente test. Para comparar modificaciones,
reservaremos validación dentro de train en una sesión posterior.

**Respuesta (Checkpoint 4):**
- **Falso positivo:** pasajero real 0 al que el modelo predijo 1 (fila «Real 0», columna «Predicción 1»): **12** casos. Predice que sobrevive, pero no sobrevivió.
- **Falso negativo:** real 1 predicho como 0: **23** casos. Predice que no sobrevive, pero sí sobrevivió. Hay casi el doble de falsos negativos que de falsos positivos, es decir, el modelo se queda corto detectando supervivientes.
- **Comparación con la base:** la base obtiene 0,615 de exactitud sin usar ninguna característica (predice siempre «no sobrevivió»). El modelo llega a 0,804, unos 19 puntos más, así que **sí aprende algo útil de las características**. Como el EDA previo usó todo Titanic, esta comparación es demostrativa, no una estimación totalmente independiente.
- **Por qué la exactitud sola no basta:** no distingue qué tipo de error se comete (aquí 23 falsos negativos frente a 12 falsos positivos), y los dos errores pueden tener costes distintos. Tampoco dice nada sobre justicia (p. ej. si falla más con ciertos grupos), ni sobre causalidad o utilidad práctica.

In [7]:
ejemplos = X_test.head(5).copy()
ejemplos["resultado_real"] = y_test.loc[ejemplos.index]
ejemplos["prediccion"] = modelo.predict(X_test.head(5))
display(ejemplos)


,Age,Fare,SibSp,Parch,Pclass,Sex,Embarked,resultado_real,prediccion
565,24.0,24.1500,2,0,3,male,S,0,0
160,44.0,16.1000,0,1,3,male,S,0,0
553,22.0,7.2250,0,0,3,male,C,1,0
860,41.0,14.1083,2,0,3,male,S,0,0
241,NaN,15.5000,1,0,3,female,Q,1,1


## 5. Pequeño trabajo propio y cierre

1. Sin reentrenar, localiza en la tabla o matriz un acierto y un tipo de error;
   explica qué significan.
2. Propón una característica alternativa y explica por qué podría o no ser útil.
   **No vuelvas a consultar test para decidir si mejora.**
3. En tu cuaderno de seguimiento anota: hecho / pendiente; qué se aprende de train,
   para qué se reserva test, una limitación y la aportación de cada persona.

No se crea una nueva entrega formal ni una nota por este primer recorrido.
Sigue las indicaciones del profesor para compartir tu trabajo.


## Respuestas al apartado 5 (trabajo propio)

**1. Un acierto y un tipo de error** (tabla de ejemplos y matriz)
- *Acierto:* el pasajero 241 (mujer, 3.ª clase, embarcó en Q) era superviviente y el modelo predijo 1. Es un verdadero positivo.
- *Error (falso negativo):* el pasajero 553 (hombre, 3.ª clase) sobrevivió, pero el modelo predijo 0. En la matriz hay 23 casos así. Significa que el modelo no reconoció a un superviviente, probablemente porque el perfil «hombre de 3.ª clase» se asocia en entrenamiento con no sobrevivir.
- *Error (falso positivo):* en la matriz hay 12 casos de pasajeros que no sobrevivieron y el modelo predijo que sí.

**2. Característica alternativa:** `TamanoFamilia = SibSp + Parch + 1`.
- *Por qué podría ser útil:* resume en una sola variable el número de acompañantes. Viajar solo o con una familia muy numerosa podría relacionarse de forma distinta con la supervivencia, y esa relación no es lineal (los extremos podrían comportarse distinto de los tamaños intermedios), cosa que `SibSp` y `Parch` por separado capturan peor.
- *Por qué podría no serlo:* es una combinación exacta de dos columnas que ya están, así que aporta poca información nueva a un modelo lineal, y podría añadir redundancia.
- No se ha reentrenado ni se ha mirado test para decidir si mejora; eso se evaluaría con validación dentro de train en una sesión posterior.

**3. Seguimiento (hecho / pendiente)**
- *Hecho:* se separó train/test con estratificación, se construyó un pipeline de preparación + regresión logística, se comparó con la base (0,615 frente a 0,804) y se interpretó la matriz de confusión.
- *Pendiente:* validación cruzada dentro de train para comparar configuraciones y probar la característica nueva.
- *Qué se aprende de train y para qué se reserva test:* de train se aprenden medianas, categorías, escalas y coeficientes; test se reserva para medir el rendimiento con pasajeros no vistos.
- *Limitación:* el EDA previo se hizo con todos los datos, así que la evaluación no es plenamente independiente; además la partición se hizo una sola vez, y un 80/20 distinto podría dar una exactitud algo diferente.
- *Aportación de cada persona:* Todo ha sido realizado por mi persona.